# Sketch2Build AI — Full Training Pipeline (RunPod, Expanded)

This notebook trains both the **Vision Encoder** (sketch-to-graph) and **Layout Diffusion Model** on RunPod GPU instances.

**Expanded training scale:**
- **20,000 synthetic samples** with 12 room types, 8 regional profiles, 5 architectural styles, multi-story
- CLIP backbone **frozen** (only train detection head — 10× faster)
- **30 epochs** vision encoder / **50 epochs** layout diffusion (early stopping enabled)
- **Batch size 16** for vision encoder, **batch size 4** for diffusion
- Data augmentation: flips, rotations, blur, brightness jitter, noise
- Mixed dataset support (RPlan + CubiCasa5K when available)

**Estimated time:** ~5-6 hours total
- Vision Encoder: ~40 min
- Layout Diffusion: ~4-5 hours

**RunPod setup:**
1. Go to [runpod.io](https://runpod.io) and create a **Pod** with a GPU (T4 ~$0.20/hr, A100 ~$0.80/hr)
2. Select a template with **PyTorch + Jupyter** (e.g., 'RunPod PyTorch 2.1+')
3. Open the Jupyter notebook in your browser
4. Upload this notebook and run all cells

**Advantages:**
- No random disconnects (pod stays running until you stop it)
- `/workspace/` storage is **persistent** — survives pod restarts
- Full training completes in one session
- ~$2-5 total cost for the full training run

## 1. Setup & Dependencies

In [ ]:
# Check GPU
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f'VRAM: {vram / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected! Make sure you selected a GPU pod template.')

In [ ]:
# Set up checkpoint directory (RunPod persistent storage)
import os

CHECKPOINT_DIR = '/workspace/sketch2build_models'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(f'{CHECKPOINT_DIR}/vision_encoder', exist_ok=True)
os.makedirs(f'{CHECKPOINT_DIR}/layout_diffusion', exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')
print('This storage is persistent — it survives pod restarts.')

In [ ]:
# Clone the repository
import os
if not os.path.exists('/workspace/sketch2build'):
    !git clone https://github.com/KachiAlex/sketch2build.git /workspace/sketch2build
else:
    print('Repo already cloned, pulling latest...')
    !cd /workspace/sketch2build && git pull

os.chdir('/workspace/sketch2build/services/ai')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies and the package itself
!pip install -q structlog pydantic-settings shapely networkx wandb transformers 2>&1 | tail -5
!pip install -q -e /workspace/sketch2build/services/ai 2>&1 | tail -5
print('Dependencies installed, package installed in editable mode.')

In [ ]:
# Set up environment variables
import os
os.environ['DATA_DIR'] = '/workspace/data'
os.environ['MODELS_DIR'] = CHECKPOINT_DIR
os.environ['OUTPUT_DIR'] = '/workspace/output'
os.environ['WANDB_PROJECT'] = 'sketch2build-ai'

os.makedirs('/workspace/data/synthetic/images', exist_ok=True)
os.makedirs('/workspace/data/synthetic/sketches', exist_ok=True)
os.makedirs('/workspace/output', exist_ok=True)

print('Environment configured.')

## 2. Generate Synthetic Training Data

Generates 20,000 synthetic floor plan + sketch pairs using the expanded graph-based generator with:
- 12 room types (living, kitchen, bedroom, bathroom, dining, hallway, entrance, storage, balcony, garage, office, utility)
- 8 regional profiles (generic, US, Nordic, Middle East, Tropical, Japan, Europe, Arid)
- 5 architectural styles (modern, traditional, compact, open_plan, luxury)
- Multi-story buildings (1-2 stories)
- 3-12 rooms per plan
- 70% valid / 30% invalid (for contrastive learning)

In [ ]:
import sys, os
sys.path.insert(0, '/workspace/sketch2build/services/ai')
sys.path.insert(0, '/workspace/sketch2build')

try:
    from src.training.data.graph_generator import GraphBasedLayoutGenerator, FloorPlan
except ImportError:
    !pip install -q -e /workspace/sketch2build/services/ai
    from src.training.data.graph_generator import GraphBasedLayoutGenerator, FloorPlan

import json, random, numpy as np
from pathlib import Path

NUM_SAMPLES = 20000
OUTPUT_DIR = Path('/workspace/data/synthetic')
IMAGE_SIZE = 256

generator = GraphBasedLayoutGenerator(seed=42)
regions = list(generator.REGIONAL_PROFILES.keys())
styles = generator.STYLES

metadata = []
for i in range(NUM_SAMPLES):
    # Diverse sampling across regions and styles
    region = random.choice(regions)
    style = random.choice(styles)
    num_rooms = random.randint(3, 12)
    is_valid = random.random() < 0.7

    plan = generator.generate(num_rooms=num_rooms, valid=is_valid, region=region, style=style)
    plan_img = plan.to_image(IMAGE_SIZE)
    sketch_img = plan.to_sketch(IMAGE_SIZE)
    img_path = OUTPUT_DIR / 'images' / f'plan_{i:06d}.png'
    sketch_path = OUTPUT_DIR / 'sketches' / f'sketch_{i:06d}.png'
    plan_img.save(img_path)
    sketch_img.save(sketch_path)
    metadata.append({
        'id': i,
        'image': f'images/plan_{i:06d}.png',
        'sketch': f'sketches/sketch_{i:06d}.png',
        'rooms': [{'type': r.type, 'x': r.x, 'y': r.y, 'w': r.width, 'd': r.depth} for r in plan.rooms],
        'adjacency': plan.adjacency,
        'valid': len(plan.rooms) >= 3 and all(r.width >= 1.5 and r.depth >= 1.5 for r in plan.rooms),
        'width': plan.plot_width,
        'depth': plan.plot_depth,
        'num_stories': plan.num_stories,
        'region': plan.region,
        'style': plan.style,
    })
    if (i + 1) % 2000 == 0:
        print(f'Generated {i+1}/{NUM_SAMPLES} samples')

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f)

print(f'\nDataset complete: {NUM_SAMPLES} samples saved to {OUTPUT_DIR}')
print(f'  - {sum(1 for m in metadata if m["valid"])} valid plans')
print(f'  - {sum(1 for m in metadata if not m["valid"])} invalid plans (for contrastive learning)')
print(f'  - Regions: { {r: sum(1 for m in metadata if m["region"]==r) for r in regions} }')
print(f'  - Multi-story: {sum(1 for m in metadata if m["num_stories"]>1)}')

In [ ]:
# Visualize a few samples
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    idx = i * 300
    plan_img = Image.open(OUTPUT_DIR / 'images' / f'plan_{idx:06d}.png')
    sketch_img = Image.open(OUTPUT_DIR / 'sketches' / f'sketch_{idx:06d}.png')
    axes[0, i].imshow(plan_img)
    axes[0, i].set_title(f'Plan #{idx}'); axes[0, i].axis('off')
    axes[1, i].imshow(sketch_img)
    axes[1, i].set_title(f'Sketch #{idx}'); axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## 3. Train Vision Encoder (Sketch → Room Graph)

- **Epochs:** 30 (with early stopping, patience=8)
- **Batch size:** 16 (frozen backbone = less memory)
- **Learning rate:** 1e-4 with cosine warmup
- **Data augmentation:** flips, rotations, blur, brightness jitter
- **Estimated time:** ~40 min

In [ ]:
USE_WANDB = False
if USE_WANDB:
    !pip install -q wandb
    import wandb
    wandb.login()
else:
    print('Skipping W&B. Set USE_WANDB=True to enable.')

In [ ]:
import os, sys, time, json
sys.path.insert(0, '/workspace/sketch2build/services/ai')
sys.path.insert(0, '/workspace/sketch2build')

try:
    import structlog
except ImportError:
    !pip install -q structlog
    import structlog

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from transformers import CLIPProcessor

from src.models.vision.sketch_encoder import SketchEncoderModel
from src.training.data.dataset import SketchDataset

structlog.get_logger().info('Starting vision encoder training')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-4
WARMUP_STEPS = 300
PATIENCE = 8
MAX_GRAD_NORM = 1.0
VAL_SPLIT = 0.1

OUTPUT_DIR = f'{CHECKPOINT_DIR}/vision_encoder'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Loading CLIP ViT-Large + cross-attention room detection head (backbone frozen)...')
model = SketchEncoderModel(
    pretrained_model='openai/clip-vit-large-patch14',
    freeze_backbone=True,
).to(DEVICE)
print(f'Model loaded on {DEVICE}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

processor = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')

print('Loading dataset (20K samples with augmentation)...')
full_dataset = SketchDataset(
    metadata_path='/workspace/data/synthetic/metadata.json',
    image_dir='/workspace/data/synthetic',
    processor=processor,
    split='train',
    max_rooms=20,
    augment=True,
)
print(f'Dataset size: {len(full_dataset)}')

val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
full_dataset.augment = False  # Will be re-enabled below
print(f'Train: {train_size}, Val: {val_size}')

In [ ]:
def collate_fn(batch):
    max_rooms = 20
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    B = len(batch)
    room_presence = torch.zeros(B, max_rooms)
    room_types = torch.full((B, max_rooms), -1, dtype=torch.long)
    bboxes = torch.zeros(B, max_rooms, 4)
    adjacency = torch.zeros(B, max_rooms, max_rooms)
    mask = torch.zeros(B, max_rooms, dtype=torch.bool)
    for b, item in enumerate(batch):
        mask[b] = item['mask']
        room_types[b] = item['room_types']
        bboxes[b] = item['bboxes']
        room_presence[b] = item['mask'].float()
        adjacency[b] = item['adjacency']
    return {
        'pixel_values': pixel_values, 'room_presence': room_presence,
        'room_types': room_types, 'bboxes': bboxes,
        'adjacency': adjacency, 'mask': mask,
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=collate_fn, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_STEPS)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS)

print(f'Total steps: {total_steps}')
print(f'Steps per epoch: {len(train_loader)}')

In [ ]:
import os, time

# Re-enable augmentation for training
full_dataset.augment = True

best_val_loss = float('inf')
patience_counter = 0
global_step = 0

ckpt_path = os.path.join(OUTPUT_DIR, 'best.pt')
if os.path.exists(ckpt_path):
    print(f'Resuming from checkpoint: {ckpt_path}')
    epoch_start, best_val_loss = model.load_checkpoint(ckpt_path, optimizer=optimizer, device=DEVICE)
    print(f'Resumed at epoch {epoch_start}, best val loss: {best_val_loss:.4f}')
else:
    epoch_start = 0
    print('Starting fresh training.')

print(f'\nStarting training for {EPOCHS} epochs...')

train_losses = []
val_losses = []

for epoch in range(epoch_start, EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    start_time = time.time()

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(DEVICE)
        targets = {k: v.to(DEVICE) for k, v in batch.items() if k != 'pixel_values'}
        optimizer.zero_grad()
        outputs = model(pixel_values)
        losses = model.compute_loss(outputs, targets)
        losses['total'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        epoch_loss += losses['total'].item()
        num_batches += 1
        global_step += 1
        if global_step <= WARMUP_STEPS:
            warmup_scheduler.step()
        else:
            cosine_scheduler.step()
        if (batch_idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            print(f'  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {losses["total"].item():.4f} | {elapsed:.0f}s')

    # Disable augmentation for validation
    full_dataset.augment = False
    model.eval()
    val_loss = 0.0
    val_batches = 0
    correct_types = 0
    total_rooms = 0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch['pixel_values'].to(DEVICE)
            targets = {k: v.to(DEVICE) for k, v in batch.items() if k != 'pixel_values'}
            outputs = model(pixel_values)
            losses = model.compute_loss(outputs, targets)
            val_loss += losses['total'].item()
            val_batches += 1
            valid = targets['room_types'] >= 0
            if valid.any():
                pred_types = outputs['room_type_logits'][valid].argmax(dim=-1)
                true_types = targets['room_types'][valid]
                correct_types += (pred_types == true_types).sum().item()
                total_rooms += valid.sum().item()

    avg_train_loss = epoch_loss / max(num_batches, 1)
    avg_val_loss = val_loss / max(val_batches, 1)
    type_acc = correct_types / max(total_rooms, 1)
    epoch_time = time.time() - start_time
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f'\nEpoch {epoch+1}/{EPOCHS} DONE | Train Loss: {avg_train_loss:.4f} | '
          f'Val Loss: {avg_val_loss:.4f} | Type Acc: {type_acc*100:.1f}% | {epoch_time:.0f}s')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        model.save_checkpoint(os.path.join(OUTPUT_DIR, 'best.pt'),
                              optimizer=optimizer, epoch=epoch+1, best_loss=best_val_loss)
        print(f'  ★ New best model saved! Val loss: {best_val_loss:.4f}')
    else:
        patience_counter += 1
        print(f'  No improvement. Patience: {patience_counter}/{PATIENCE}')

    if (epoch + 1) % 5 == 0:
        model.save_checkpoint(os.path.join(OUTPUT_DIR, f'checkpoint_epoch_{epoch+1}.pt'),
                              optimizer=optimizer, epoch=epoch+1, best_loss=best_val_loss)

    # Re-enable augmentation for next training epoch
    full_dataset.augment = True

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}! Best val loss: {best_val_loss:.4f}')
        break

print(f'\n✅ Vision encoder training complete! Best val loss: {best_val_loss:.4f}')
print(f'Checkpoints saved to: {OUTPUT_DIR}')

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, label='Train Loss', color='blue')
ax1.plot(val_losses, label='Val Loss', color='red')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Vision Encoder — Training Loss'); ax1.legend(); ax1.grid(True)
ax2.plot(train_losses, label='Train', color='blue')
ax2.plot(val_losses, label='Val', color='red')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('Loss (log scale)'); ax2.set_yscale('log'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/vision_encoder/training_curve.png')
plt.show()
print('Training curve saved.')

## 4. Train Layout Diffusion Model (Room Graph → Floor Plan)

- **Epochs:** 50 (with early stopping, patience=12)
- **Batch size:** 4 (U-Net is memory-heavy)
- **Data augmentation:** horizontal flips with bbox mirroring, noise injection
- **Multi-story conditioning:** model receives num_stories as conditioning
- **Estimated time:** ~4-5 hours

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
import numpy as np
import time, os, sys

sys.path.insert(0, '/workspace/sketch2build/services/ai')
sys.path.insert(0, '/workspace/sketch2build')

try:
    import structlog
except ImportError:
    !pip install -q structlog

from src.models.layout.diffusion import LayoutDiffusionModel
from src.training.data.dataset import LayoutDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-4
WARMUP_STEPS = 300
PATIENCE = 12
MAX_GRAD_NORM = 1.0
VAL_SPLIT = 0.1

OUTPUT_DIR = f'{CHECKPOINT_DIR}/layout_diffusion'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Creating Layout Diffusion Model (U-Net + DDPM with multi-story conditioning)...')
model = LayoutDiffusionModel(num_train_timesteps=1000).to(DEVICE)
print(f'Model on {DEVICE}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

print('Loading dataset (20K samples with augmentation)...')
full_dataset = LayoutDataset(
    metadata_path='/workspace/data/synthetic/metadata.json',
    image_dir='/workspace/data/synthetic',
    split='train',
    image_size=256,
    augment=True,
)
print(f'Dataset size: {len(full_dataset)}')

val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
full_dataset.augment = False  # Will be re-enabled in training loop
print(f'Train: {train_size}, Val: {val_size}')

In [ ]:
def layout_collate_fn(batch):
    max_rooms = 20
    images = torch.stack([item['image'] for item in batch])
    B = len(batch)
    room_types = torch.full((B, max_rooms), 0, dtype=torch.long)
    bboxes = torch.zeros(B, max_rooms, 4)
    adjacency = torch.zeros(B, max_rooms, max_rooms)
    mask = torch.zeros(B, max_rooms, dtype=torch.bool)
    num_stories = torch.zeros(B, dtype=torch.long)
    for b, item in enumerate(batch):
        rooms = item.get('rooms', [])
        for i, room in enumerate(rooms[:max_rooms]):
            room_types[b, i] = room.get('type_idx', 0)
            bbox = room.get('bbox', [0, 0, 0, 0])
            if len(bbox) == 4:
                bboxes[b, i] = torch.tensor(bbox, dtype=torch.float32)
            mask[b, i] = True
        adj = item.get('adjacency', [])
        for pair in adj:
            if isinstance(pair, (list, tuple)) and len(pair) >= 2:
                a, c = int(pair[0]), int(pair[1])
                if a < max_rooms and c < max_rooms:
                    adjacency[b, a, c] = 1.0
                    adjacency[b, c, a] = 1.0
        num_stories[b] = item.get('num_stories', 1)
    return {
        'images': images, 'room_types': room_types, 'bboxes': bboxes,
        'adjacency': adjacency, 'mask': mask, 'num_stories': num_stories,
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=layout_collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=layout_collate_fn, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_STEPS)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS)

print(f'Total steps: {total_steps}')
print(f'Steps per epoch: {len(train_loader)}')
print(f'Estimated time: ~{len(train_loader) * EPOCHS * 0.5 / 3600:.1f} hours')

In [ ]:
import os, time

# Re-enable augmentation for training
full_dataset.augment = True

best_val_loss = float('inf')
patience_counter = 0
global_step = 0

ckpt_path = os.path.join(OUTPUT_DIR, 'best.pt')
if os.path.exists(ckpt_path):
    print(f'Resuming from checkpoint: {ckpt_path}')
    epoch_start, best_val_loss = model.load_checkpoint(ckpt_path, optimizer=optimizer, device=DEVICE)
    print(f'Resumed at epoch {epoch_start}, best val loss: {best_val_loss:.4f}')
else:
    epoch_start = 0
    print('Starting fresh training.')

print(f'\nStarting layout diffusion training for {EPOCHS} epochs...')

layout_train_losses = []
layout_val_losses = []

for epoch in range(epoch_start, EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    start_time = time.time()

    for batch_idx, batch in enumerate(train_loader):
        images = batch['images'].to(DEVICE)
        room_types = batch['room_types'].to(DEVICE)
        bboxes = batch['bboxes'].to(DEVICE)
        adjacency = batch['adjacency'].to(DEVICE)
        mask = batch['mask'].to(DEVICE)
        num_stories = batch['num_stories'].to(DEVICE)
        optimizer.zero_grad()
        loss = model(images, room_types, bboxes, adjacency, mask, num_stories=num_stories)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        epoch_loss += loss.item()
        num_batches += 1
        global_step += 1
        if global_step <= WARMUP_STEPS:
            warmup_scheduler.step()
        else:
            cosine_scheduler.step()
        if (batch_idx + 1) % 200 == 0:
            elapsed = time.time() - start_time
            print(f'  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {loss.item():.4f} | {elapsed:.0f}s')

    # Disable augmentation for validation
    full_dataset.augment = False
    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['images'].to(DEVICE)
            room_types = batch['room_types'].to(DEVICE)
            bboxes = batch['bboxes'].to(DEVICE)
            adjacency = batch['adjacency'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            num_stories = batch['num_stories'].to(DEVICE)
            loss = model(images, room_types, bboxes, adjacency, mask, num_stories=num_stories)
            val_loss += loss.item()
            val_batches += 1

    avg_train_loss = epoch_loss / max(num_batches, 1)
    avg_val_loss = val_loss / max(val_batches, 1)
    epoch_time = time.time() - start_time
    layout_train_losses.append(avg_train_loss)
    layout_val_losses.append(avg_val_loss)

    print(f'\nEpoch {epoch+1}/{EPOCHS} DONE | Train Loss: {avg_train_loss:.4f} | '
          f'Val Loss: {avg_val_loss:.4f} | {epoch_time:.0f}s')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        model.save_checkpoint(os.path.join(OUTPUT_DIR, 'best.pt'),
                              optimizer=optimizer, epoch=epoch+1, best_loss=best_val_loss)
        print(f'  ★ New best model saved! Val loss: {best_val_loss:.4f}')
    else:
        patience_counter += 1
        print(f'  No improvement. Patience: {patience_counter}/{PATIENCE}')

    if (epoch + 1) % 10 == 0:
        model.save_checkpoint(os.path.join(OUTPUT_DIR, f'checkpoint_epoch_{epoch+1}.pt'),
                              optimizer=optimizer, epoch=epoch+1, best_loss=best_val_loss)
        model.eval()
        with torch.no_grad():
            sample = model.generate(room_types=room_types[:1], bboxes=bboxes[:1],
                                    adjacency=adjacency[:1], mask=mask[:1],
                                    shape=(1, 3, 256, 256), num_inference_steps=20,
                                    num_stories=num_stories[:1])
            sample_img = sample[0].cpu()
            sample_img = ((sample_img + 1) / 2 * 255).clamp(0, 255).byte()
            from PIL import Image
            Image.fromarray(sample_img.permute(1, 2, 0).numpy()).save(
                os.path.join(OUTPUT_DIR, f'sample_epoch_{epoch+1}.png'))
            print(f'  Sample saved: sample_epoch_{epoch+1}.png')

    # Re-enable augmentation for next training epoch
    full_dataset.augment = True

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}! Best val loss: {best_val_loss:.4f}')
        break

print(f'\n✅ Layout diffusion training complete! Best val loss: {best_val_loss:.4f}')
print(f'Checkpoints saved to: {OUTPUT_DIR}')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(layout_train_losses, label='Train Loss', color='blue')
ax.plot(layout_val_losses, label='Val Loss', color='red')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Layout Diffusion — Training Loss'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/layout_diffusion/training_curve.png')
plt.show()
print('Training curve saved.')

## 5. Generate Sample Designs

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image

model.eval()
# Use normalized coordinates (0-1 range) matching training data
room_types = torch.tensor([[0, 1, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.long).to(DEVICE)
bboxes = torch.tensor([[[0.0, 0.0, 0.4, 0.3], [0.4, 0.0, 0.25, 0.2], [0.0, 0.3, 0.3, 0.25], [0.3, 0.3, 0.15, 0.15]] + [[0,0,0,0]]*16], dtype=torch.float32).to(DEVICE)
adjacency = torch.zeros(1, 20, 20, dtype=torch.float32).to(DEVICE)
adjacency[0, 0, 1] = adjacency[0, 1, 0] = 1
adjacency[0, 0, 2] = adjacency[0, 2, 0] = 1
adjacency[0, 1, 3] = adjacency[0, 3, 1] = 1
mask = torch.tensor([[True, True, True, True] + [False]*16], dtype=torch.bool).to(DEVICE)
num_stories = torch.tensor([1], dtype=torch.long).to(DEVICE)

print('Generating 4 floor plan samples...')
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i in range(4):
    with torch.no_grad():
        sample = model.generate(room_types=room_types, bboxes=bboxes, adjacency=adjacency,
                                mask=mask, shape=(1, 3, 256, 256), num_inference_steps=50,
                                num_stories=num_stories)
    img = sample[0].cpu()
    img = ((img + 1) / 2 * 255).clamp(0, 255).byte()
    axes[i].imshow(img.permute(1, 2, 0).numpy())
    axes[i].set_title(f'Sample {i+1}'); axes[i].axis('off')
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/layout_diffusion/generated_samples.png')
plt.show()
print('Generated samples saved.')

## 6. Download Checkpoints

Checkpoints are in `/workspace/sketch2build_models/`:
- `vision_encoder/best.pt`
- `layout_diffusion/best.pt`

Use RunPod's file browser or `scp` to download them to your local machine.
Place them in your project at:
- `services/ai/models/vision_encoder/best.pt`
- `services/ai/models/layout_diffusion/best.pt`

**Important:** Stop the pod after downloading to save costs. The `/workspace/` storage persists even when the pod is stopped.

In [ ]:
import os

print('=== Vision Encoder Checkpoints ===')
for f in sorted(os.listdir(f'{CHECKPOINT_DIR}/vision_encoder')):
    size = os.path.getsize(os.path.join(f'{CHECKPOINT_DIR}/vision_encoder', f)) / 1e6
    print(f'  {f} ({size:.1f} MB)')

print('\n=== Layout Diffusion Checkpoints ===')
for f in sorted(os.listdir(f'{CHECKPOINT_DIR}/layout_diffusion')):
    size = os.path.getsize(os.path.join(f'{CHECKPOINT_DIR}/layout_diffusion', f)) / 1e6
    print(f'  {f} ({size:.1f} MB)')

print(f'\n✅ All checkpoints are in: {CHECKPOINT_DIR}')